In [1]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering, KMeans
from scipy.cluster.hierarchy import linkage, dendrogram

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict, Counter

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")



Libraries loaded successfully


In [2]:
file_path = r'C:\Users\Liza\Documents\Kerja Praktik\Topic modeling\berttopic\exp-5\bertopic_results(5).csv'
df = pd.read_csv(file_path, sep=';')


In [3]:
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopwords_indonesia = stopword_factory.get_stop_words()

additional_stopwords = [
    # pronomina
    'aku','saya','kamu','dia','mereka','kita','kalian','orang','orang-orang','sama-sama',

    # kata_tanya_umum
    'cara','gimana','bagaimana','mana','apa','kenapa','ngapain','ngapa','yang','sih','tanya',
    'saja','dong','gitu','kayak','aja','cuma','kayaknya','begitu', 'kalau', 'sama', 'jadi', 

    # kata_verba_umum
    'coba','bikin','buat','biar','lihat','lihat-lihat','ajar','lihatnya','punya','buatnya',
    'buatkan','cari','temu','ketemu','bilang','ucap','cerita','ceritain','tulis','ganti',
    'ambil','tarik','berasa', 'memang', 'kira', 'alih', 'nuna', 'ikut', 'gambar', 'suara', 'nyanyi'
]
stopwords_indonesia = set(stopwords_indonesia).union(set(additional_stopwords))

In [ ]:
### try 2
print("Loading Transformer model...")
embedding_model = SentenceTransformer("indobenchmark/indobert-base-p2")

print("Generating embeddings...")
documents = df['document_preprocessed'].tolist()
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"Embeddings: {embeddings.shape}")

np.save("embeddings.npy", embeddings)


Loading Transformer model...


No sentence-transformers model found with name indobenchmark/indobert-base-p2. Creating a new one with mean pooling.


Generating embeddings...


Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Embeddings: (7500, 768)


In [7]:
SEED = 42

# Seed
import random, numpy as np, torch, os
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Deterministic
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Models
umap_model = UMAP(
    n_neighbors=11,
    n_components=6,
    min_dist=0.0,
    metric='cosine',
    random_state=SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=28,
    min_samples=7,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords_indonesia),
    min_df=2,
    ngram_range=(1, 2)
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents, embeddings)

2025-12-24 21:16:58,068 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2025-12-24 21:17:56,511 - BERTopic - Dimensionality - Completed ✓
2025-12-24 21:17:56,517 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-24 21:17:58,518 - BERTopic - Cluster - Completed ✓
2025-12-24 21:17:58,518 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-12-24 21:17:58,978 - BERTopic - Representation - Completed ✓
2025-12-24 21:17:58,978 - BERTopic - Topic reduction - Reducing number of topics
2025-12-24 21:17:59,010 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-24 21:17:59,355 - BERTopic - Representation - Completed ✓
2025-12-24 21:17:59,360 - BERTopic - Topic reduction - Reduced number of topics from 58 to 15


In [8]:
topic_info = topic_model.get_topic_info()
n_topics = len(topic_info[topic_info['Topic'] != -1])
outlier_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if -1 in topic_info['Topic'].values else 0

print(f"\nInitial Results:")
print(f"Topics: {n_topics}")
print(f"Outliers: {outlier_count:,} ({outlier_count/len(df)*100:.2f}%)")


Initial Results:
Topics: 14
Outliers: 3,226 (43.01%)


## 6. Prepare Dataframe

In [9]:
df['Topic'] = topics
if probs is not None and len(probs.shape) > 1:
    df['Probability'] = probs.max(axis=1)
else:
    df['Probability'] = probs

topic_words = {}
for topic_id in df['Topic'].unique():
    if topic_id == -1:
        topic_words[topic_id] = "outlier"
    else:
        words = topic_model.get_topic(topic_id)
        if words:
            top_words = [word for word, _ in words[:10]]
            topic_words[topic_id] = ' - '.join(top_words)
        else:
            topic_words[topic_id] = "no_words"

df['Top_n_words'] = df['Topic'].map(topic_words)
topic_names = {info['Topic']: info['Name'] for _, info in topic_info.iterrows()}
df['Name'] = df['Topic'].map(topic_names)
df['Document'] = df['document_preprocessed']

print("DataFrame prepared")

DataFrame prepared


In [10]:
summary = (
    df.groupby('Topic')
      .agg(
          Total_Data=('Topic', 'count'),
          Top_N_Words=('Top_n_words', 'first')
      )
      .reset_index()
      .sort_values('Topic')
)

summary

,Topic,Total_Data,Top_N_Words
0,-1,3226,outlier
1,0,3089,diri - sendiri - kecil - hidup - ngerasa - tak...
2,1,337,psikolog - terapi - takut - khawatir - dengeri...
3,2,302,gagal - takut gagal - takut - kecil - sukses -...
4,3,95,lupa - ingat - ulang - otak - materi - simpan ...
5,4,83,overthinking - henti - capek - overthinking ca...
6,5,70,ubah - kecil - ubah kecil - kecil ubah - hidup...
7,6,62,anak - puji - main - salah - anak takut - mama...
8,7,50,dokter - takut dokter - gila - takut - jiwa - ...
9,8,33,ulang - kompulsif - obsesif kompulsif - ganggu...


In [11]:
df.to_csv("before_reassign_fixindobert.csv", sep=";", index=False)
df

,question,answer,from,filename,document_preprocessed,Topic,Probability,Top_n_words,Name,Document
0,Apa itu memvalidasi pikiran sendiri?,Cek data yang mendukung dan yang menolak keyak...,GPT,A1_GPT_EmotionalFirstAid23-45_50.json,memvalidasi sendiri data dukung tolak yakin ga...,-1,0.251203,outlier,-1_kecil_diri_sendiri_hidup,memvalidasi sendiri data dukung tolak yakin ga...
1,Kenapa aku malas mencoba lagi setelah jatuh?,"Kegagalan menciptakan ilusi ""hasilnya pasti sa...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,malas jatuh gagal cipta ilusi hasil alternatif...,0,0.673766,diri - sendiri - kecil - hidup - ngerasa - tak...,0_diri_sendiri_kecil_hidup,malas jatuh gagal cipta ilusi hasil alternatif...
2,Bagaimana cara efektif meminta bantuan?,"Ungkap kebutuhan spesifik: ""Aku butuh teman br...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,efektif minta ungkap butuh spesifik butuh tema...,0,0.694161,diri - sendiri - kecil - hidup - ngerasa - tak...,0_diri_sendiri_kecil_hidup,efektif minta ungkap butuh spesifik butuh tema...
3,Apa saja contoh faktor yang bisa kukendalikan?,"Jumlah waktu belajar harian, mentor yang dihub...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,contoh faktor kendali jumlah mentor target min...,10,1.000000,kuesioner - metode - eksperimental - metode ek...,10_kuesioner_metode_eksperimental_metode ekspe...,contoh faktor kendali jumlah mentor target min...
4,Bagaimana memecah tujuan besar agar realistis?,"Gunakan metode SMART-ER (Specific, Measurable,...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,pecah tuju realistis metode smart specific mea...,10,1.000000,kuesioner - metode - eksperimental - metode ek...,10_kuesioner_metode_eksperimental_metode ekspe...,pecah tuju realistis metode smart specific mea...
...,...,...,...,...,...,...,...,...,...,...
7495,Aku sering merasa orang lain nggak ngerti beta...,Pasti frustrasi kalau usahamu nggak diakui. Ta...,GROK,D9_GROK_EgoIsTheEnemy139-149_20.json,betapa usaha salah frustrasi usaha terlalu ene...,0,0.678011,diri - sendiri - kecil - hidup - ngerasa - tak...,0_diri_sendiri_kecil_hidup,betapa usaha salah frustrasi usaha terlalu ene...
7496,Aku sering kesel sama tim karena mereka nggak ...,Setiap orang punya ritme kerja sendiri. Coba k...,GROK,D9_GROK_EgoIsTheEnemy184-194_20.json,kesel secepet salah ritme kerja sendiri target...,2,0.238851,gagal - takut gagal - takut - kecil - sukses -...,2_gagal_takut gagal_takut_kecil,kesel secepet salah ritme kerja sendiri target...
7497,Kenapa aku susah buat konsisten sama tujuanku?,"Kadang, kita kehilangan fokus karena terlalu m...",GROK,D9_GROK_EgoIsTheEnemy39-49_20.json,susah konsisten tuju hilang terlalu mikirin ha...,2,0.273361,gagal - takut gagal - takut - kecil - sukses -...,2_gagal_takut gagal_takut_kecil,susah konsisten tuju hilang terlalu mikirin ha...
7498,"Aku pengen jadi orang hebat, tapi kok kayaknya...",Jadi orang hebat nggak selalu tentang push dir...,GROK,D9_GROK_EgoIsTheEnemy61-71_20.json,jauh push diri mati mati push diri burnout kec...,5,0.229201,ubah - kecil - ubah kecil - kecil ubah - hidup...,5_ubah_kecil_ubah kecil_kecil ubah,jauh push diri mati mati push diri burnout kec...


## 7. Evaluation Function

In [12]:
def topic_words_from_dataframe(df_input, top_n=10):
    """Extract topic keywords from DataFrame with error handling"""
    df_valid = df_input[df_input["Topic"] != -1].copy()
    topic_words = {}

    for topic_id in sorted(df_valid["Topic"].unique()):
        try:
            top_words_str = df_valid[df_valid["Topic"] == topic_id]["Top_n_words"].iloc[0]

            # Handle different formats and empty values
            if pd.isna(top_words_str) or top_words_str in ["", "outlier", "no_words"]:
                continue

            words = [w.strip() for w in str(top_words_str).split(" - ") if w.strip() and w.strip() not in ["outlier", "no_words", ""]]

            if words:  # Only add if we have valid words
                topic_words[topic_id] = words[:top_n]
        except Exception as e:
            continue

    return topic_words

def compute_topic_diversity(topic_words_dict):
    """Calculate topic diversity (unique words / total words)"""
    if not topic_words_dict:
        return 0.0
    all_words = [w for words in topic_words_dict.values() for w in words]
    if not all_words:
        return 0.0
    return len(set(all_words)) / len(all_words)

def compute_coherence_cv(docs_tokenized, topic_words_dict):
    """Calculate coherence C_v using Gensim with error handling"""
    if not topic_words_dict or not docs_tokenized:
        return 0.0

    # Filter out topics with empty/invalid words
    valid_topics = []
    for topic_id, words in topic_words_dict.items():
        if words and all(isinstance(w, str) and w.strip() for w in words):
            valid_topics.append(words)

    if not valid_topics:
        return 0.0

    try:
        dictionary = Dictionary(docs_tokenized)
        corpus = [dictionary.doc2bow(tokens) for tokens in docs_tokenized]

        if not corpus:
            return 0.0

        cm = CoherenceModel(
            topics=valid_topics,
            texts=docs_tokenized,
            dictionary=dictionary,
            corpus=corpus,
            coherence="c_v"
        )
        return cm.get_coherence()
    except Exception as e:
        print(f"  [Warning] Coherence calculation failed: {str(e)[:50]}")
        return 0.0

def evaluate_metrics(df_input, stage_name=""):
    """Comprehensive evaluation of topic modeling metrics"""
    # Only use valid documents (not outliers)
    df_valid_docs = df_input[df_input["Topic"] != -1].copy()

    if len(df_valid_docs) == 0:
        sep_line = "=" * 70
        print(f"\n{sep_line}")
        print(f"Evaluation: {stage_name}")
        print(sep_line)
        print("ERROR: No valid documents")
        print(sep_line)
        return {"stage": stage_name, "coherence": 0, "diversity": 0, "n_topics": 0, "n_outliers": len(df_input), "outlier_ratio": 100}

    docs_tokenized = [doc.split() for doc in df_valid_docs["Document"].tolist()]
    topic_words = topic_words_from_dataframe(df_input, top_n=10)

    if not topic_words:
        diversity = 0.0
        coherence_cv = 0.0
    else:
        diversity = compute_topic_diversity(topic_words)
        coherence_cv = compute_coherence_cv(docs_tokenized, topic_words)

    n_outliers = len(df_input[df_input["Topic"] == -1])
    outlier_ratio = n_outliers / len(df_input) * 100
    n_topics = len(topic_words)

    sep_line = "=" * 70
    print(f"\n{sep_line}")
    print(f"Evaluation: {stage_name}")
    print(sep_line)
    print(f"Coherence C_v : {coherence_cv:.4f}")
    print(f"Diversity     : {diversity:.4f}")
    print(f"Topics        : {n_topics}")
    print(f"Outliers      : {n_outliers:,} ({outlier_ratio:.2f}%)")
    print(sep_line)

    return {"stage": stage_name, "coherence": coherence_cv, "diversity": diversity, "n_topics": n_topics, "n_outliers": n_outliers, "outlier_ratio": outlier_ratio}

print("Evaluation functions defined")

Evaluation functions defined


In [13]:
metrics_initial = evaluate_metrics(df, "Initial Results")


Evaluation: Initial Results
Coherence C_v : 0.6380
Diversity     : 0.8714
Topics        : 14
Outliers      : 3,226 (43.01%)


## 8. Compute Topic Centroid

In [14]:
def compute_topic_centroids(df_input, embeddings_array, max_docs=100):
    df_valid = df_input[df_input['Topic'] != -1].copy()
    topic_centroids = {}
    for topic_id in sorted(df_valid['Topic'].unique()):
        topic_indices = df_valid[df_valid['Topic'] == topic_id].index.tolist()
        if len(topic_indices) > max_docs:
            topic_indices = np.random.choice(topic_indices, max_docs, replace=False).tolist()
        topic_embeddings = embeddings_array[topic_indices]
        centroid = np.mean(topic_embeddings, axis=0)
        topic_centroids[topic_id] = centroid
    print(f"Computed centroids for {len(topic_centroids)} topics")
    return topic_centroids

topic_centroids = compute_topic_centroids(df, embeddings, max_docs=100)

Computed centroids for 14 topics


## 9. Outlier Reassignment

In [15]:
def reassign_outliers_twostage(df_outliers, outlier_embeddings, topic_centroids, threshold_high=0.70, threshold_medium=0.60):
    """
    Reassign outliers to nearest topics using two-stage threshold
    """
    print(f"\n{'='*70}")
    print(f"Outlier Reassignment")
    print(f"{'='*70}")
    print(f"High confidence threshold  : {threshold_high}")
    print(f"Medium confidence threshold: {threshold_medium}")
    print(f"Total outliers             : {len(df_outliers):,}")

    reassignments = []
    stage1_count = 0
    stage2_count = 0

    for idx, outlier_emb in enumerate(outlier_embeddings):
        max_sim = -1
        best_topic = -1

        for topic_id, centroid in topic_centroids.items():
            sim = cosine_similarity([outlier_emb], [centroid])[0][0]
            if sim > max_sim:
                max_sim = sim
                best_topic = topic_id

        # Stage 1: High confidence 
        if max_sim >= threshold_high:
            new_topic = best_topic
            stage1_count += 1
            reassigned = True
        # Stage 2: Medium confidence
        elif max_sim >= threshold_medium:
            new_topic = best_topic
            stage2_count += 1
            reassigned = True
        else:
            new_topic = -1
            reassigned = False

        reassignments.append({
            'index': df_outliers.iloc[idx].name,
            'question': df_outliers.iloc[idx]['question'],
            'document': df_outliers.iloc[idx]['Document'],
            'old_topic': -1,
            'new_topic': new_topic,
            'similarity': max_sim,
            'reassigned': reassigned,
            'stage': 'high' if max_sim >= threshold_high else ('medium' if reassigned else 'rejected')
        })

    df_reassigned = pd.DataFrame(reassignments)
    total_reassigned = df_reassigned['reassigned'].sum()
    still_outlier = len(df_reassigned) - total_reassigned

    print(f"\n{'='*70}")
    print(f"Reassignment Results")
    print(f"{'='*70}")
    print(f"Stage 1 (High confidence)   : {stage1_count:,} ({stage1_count/len(df_reassigned)*100:.1f}%)")
    print(f"Stage 2 (Medium confidence) : {stage2_count:,} ({stage2_count/len(df_reassigned)*100:.1f}%)")
    print(f"Total reassigned            : {total_reassigned:,} ({total_reassigned/len(df_reassigned)*100:.1f}%)")
    print(f"Still outliers              : {still_outlier:,} ({still_outlier/len(df_reassigned)*100:.1f}%)")
    print(f"Average similarity          : {df_reassigned['similarity'].mean():.4f}")
    print(f"{'='*70}")

    return df_reassigned

print("Two-stage reassignment function defined")

Two-stage reassignment function defined


## Threshold Section

In [16]:
#for find_optimal_threshold_real
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

def get_topic_words(df, n_top_words=10):
    """
    Recalculate topic words after reassignment.
    Returns dict: topic_id -> list of top-N words
    """
    topic_words = {}
    vectorizer = CountVectorizer()
    docs = df['Document'].tolist()
    X = vectorizer.fit_transform(docs)
    vocab = np.array(vectorizer.get_feature_names_out())

    for topic_id in sorted(df['Topic'].unique()):
        if topic_id == -1:
            continue
        topic_docs = df[df['Topic'] == topic_id].index.tolist()
        if not topic_docs:
            topic_words[topic_id] = []
            continue

        topic_matrix = X[topic_docs].sum(axis=0)  
        top_idx = np.asarray(topic_matrix).ravel().argsort()[::-1][:n_top_words]
        topic_words[topic_id] = vocab[top_idx].tolist()

    return topic_words


In [17]:
def find_optimal_threshold_real(
    df_original,
    df_outlier,
    outlier_embeddings,
    topic_centroids,
    thresholds=[0.60, 0.65, 0.70, 0.75],
    n_top_words=10,
    w_assign=0.5,
    w_coherence=0.3,
    w_diversity=0.2
):
   
    results = []
    embeddings_size = len(df_original)

    for thr in thresholds:
        print(f"\nTesting threshold {thr} ...")

        df_test = df_original.copy()
        reassigned_count = 0

        # --- Step 1: simulate reassignment ---
        for i, out_emb in enumerate(outlier_embeddings):
            best_topic = -1
            max_sim = -1

            for topic_id, centroid in topic_centroids.items():
                sim = cosine_similarity([out_emb], [centroid])[0][0]
                if sim > max_sim:
                    max_sim = sim
                    best_topic = topic_id

            if max_sim >= thr:
                df_test.loc[df_outlier.index[i], "Topic"] = best_topic
                reassigned_count += 1

        n_outliers_remaining = (df_test["Topic"] == -1).sum()
        outlier_ratio = n_outliers_remaining / embeddings_size * 100

        # --- Step 2: recompute topic words ---
        topic_words = get_topic_words(df_test, n_top_words=n_top_words)

        # --- Step 3: recompute REAL diversity ---
        true_div = compute_topic_diversity(topic_words)

        # --- Step 4: recompute REAL coherence ---
        df_valid = df_test[df_test["Topic"] != -1]
        docs_tokenized = [t.split() for t in df_valid["Document"].tolist()]

        true_coh = compute_coherence_cv(docs_tokenized, topic_words)

        print(f"  Reassigned: {reassigned_count}")
        print(f"  Remaining outliers: {n_outliers_remaining} ({outlier_ratio:.2f}%)")
        print(f"  Diversity: {true_div:.4f}")
        print(f"  Coherence: {true_coh:.4f}")

        results.append({
            "threshold": thr,
            "reassigned": reassigned_count,
            "outliers_remaining": n_outliers_remaining,
            "diversity": true_div,
            "coherence": true_coh
        })

    df_res = pd.DataFrame(results)

    # --- Normalization ---
    df_res["assign_norm"] = df_res["reassigned"] / df_res["reassigned"].max()
    df_res["coh_norm"] = (df_res["coherence"] - df_res["coherence"].min()) / (df_res["coherence"].max() - df_res["coherence"].min() + 1e-12)
    df_res["div_norm"] = (df_res["diversity"] - df_res["diversity"].min()) / (df_res["diversity"].max() - df_res["diversity"].min() + 1e-12)

    # --- Weighted score ---
    df_res["score"] = (
          w_assign * df_res["assign_norm"]
        + w_coherence * df_res["coh_norm"]
        + w_diversity * df_res["div_norm"]
    )

    best_row = df_res.loc[df_res["score"].idxmax()]

    print("\n" + "="*70)
    print("Optimal threshold selected:")
    print(f"Threshold       : {best_row['threshold']}")
    print(f"Assigned outlier: {best_row['reassigned']}")
    print(f"Diversity       : {best_row['diversity']:.4f}")
    print(f"Coherence       : {best_row['coherence']:.4f}")
    print(f"Score           : {best_row['score']:.4f}")
    print("="*70)

    return best_row["threshold"], df_res

# Find optimal threshold
df_outlier = df[df['Topic'] == -1].copy()
outlier_indices = df_outlier.index.tolist()
outlier_embeddings = embeddings[outlier_indices]


optimal_threshold, threshold_results = find_optimal_threshold_real(
    df_original=df,
    df_outlier=df_outlier,
    outlier_embeddings=outlier_embeddings,
    topic_centroids=topic_centroids
)



Testing threshold 0.6 ...
  Reassigned: 3212
  Remaining outliers: 14 (0.19%)
  Diversity: 0.5714
  Coherence: 0.4784

Testing threshold 0.65 ...
  Reassigned: 3177
  Remaining outliers: 49 (0.65%)
  Diversity: 0.5786
  Coherence: 0.4724

Testing threshold 0.7 ...
  Reassigned: 2978
  Remaining outliers: 248 (3.31%)
  Diversity: 0.5714
  Coherence: 0.4729

Testing threshold 0.75 ...
  Reassigned: 2324
  Remaining outliers: 902 (12.03%)
  Diversity: 0.5714
  Coherence: 0.4811

Optimal threshold selected:
Threshold       : 0.6
Assigned outlier: 3212.0
Diversity       : 0.5714
Coherence       : 0.4784
Score           : 0.7064


In [18]:
# Execute reassignment with optimal threshold
df_reassigned = reassign_outliers_twostage(
    df_outlier,
    outlier_embeddings,
    topic_centroids,
    threshold_high=optimal_threshold,
    threshold_medium=max(0.60, optimal_threshold - 0.10)
)


Outlier Reassignment
High confidence threshold  : 0.6
Medium confidence threshold: 0.6
Total outliers             : 3,226

Reassignment Results
Stage 1 (High confidence)   : 3,212 (99.6%)
Stage 2 (Medium confidence) : 0 (0.0%)
Total reassigned            : 3,212 (99.6%)
Still outliers              : 14 (0.4%)
Average similarity          : 0.7744


## 11. Update Dataset

In [19]:
df_updated = df.copy()
topics_with_new_docs = set()

for idx, row in df_reassigned[df_reassigned['reassigned']].iterrows():
    df_updated.loc[row['index'], 'Topic'] = row['new_topic']
    topics_with_new_docs.add(row['new_topic'])

print(f"Topics that received outliers: {len(topics_with_new_docs)}")
print(f"Topics preserved: {metrics_initial['n_topics'] - len(topics_with_new_docs)}")

metrics_after_reassign = evaluate_metrics(df_updated, "After Outlier Reassignment")

Topics that received outliers: 14
Topics preserved: 0

Evaluation: After Outlier Reassignment
Coherence C_v : 0.3150
Diversity     : 1.0000
Topics        : 1
Outliers      : 14 (0.19%)


## Representative Word Calculation

In [20]:
def recalculate_representative_words_minimal(df_input, df_original, topics_to_recalc, min_new_docs_ratio=0.15, top_n=10):
    """
    Selectively recalculate representative words for impacted topics only
    """
    print(f"\n{'='*70}")
    print(f"Selective Recalculation")
    print(f"{'='*70}")
    print(f"Minimum new docs ratio for recalc: {min_new_docs_ratio*100:.0f}%")

    df_valid = df_input[df_input['Topic'] != -1].copy()
    vectorizer = CountVectorizer(stop_words=list(stopwords_indonesia), max_features=1000)

    new_top_words = {}
    actually_recalculated = 0

    for topic_id in sorted(df_valid['Topic'].unique()):
        # Check if this topic received new docs
        if topic_id not in topics_to_recalc:
            old_words = df_original[df_original['Topic'] == topic_id]['Top_n_words'].iloc[0]
            new_top_words[topic_id] = [w.strip() for w in old_words.split(' - ')][:top_n]
            continue

        # Check impact ratio: how many new docs vs total docs in topic
        current_size = len(df_valid[df_valid['Topic'] == topic_id])
        original_size = len(df_original[df_original['Topic'] == topic_id])
        new_docs_count = current_size - original_size

        if new_docs_count <= 0:
            impact_ratio = 0
        else:
            impact_ratio = new_docs_count / current_size

        # Only recalculate if impact is significant
        if impact_ratio < min_new_docs_ratio:
            old_words = df_original[df_original['Topic'] == topic_id]['Top_n_words'].iloc[0]
            new_top_words[topic_id] = [w.strip() for w in old_words.split(' - ')][:top_n]
            print(f"  Topic {topic_id}: Impact {impact_ratio*100:.1f}% < threshold, KEEPING original")
            continue

        # Recalculate for high-impact topics
        topic_docs = df_valid[df_valid['Topic'] == topic_id]['Document'].tolist()

        if len(topic_docs) < 3:
            old_words = df_original[df_original['Topic'] == topic_id]['Top_n_words'].iloc[0]
            new_top_words[topic_id] = [w.strip() for w in old_words.split(' - ')][:top_n]
            continue

        try:
            doc_term_matrix = vectorizer.fit_transform(topic_docs)
            tfidf_transformer = TfidfTransformer()
            tfidf_matrix = tfidf_transformer.fit_transform(doc_term_matrix)
            avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
            feature_names = vectorizer.get_feature_names_out()
            top_indices = avg_tfidf.argsort()[-top_n:][::-1]
            top_words = [feature_names[i] for i in top_indices]

            new_top_words[topic_id] = top_words
            actually_recalculated += 1
            print(f"  Topic {topic_id}: Impact {impact_ratio*100:.1f}%, RECALCULATED ({len(topic_docs)} docs, +{new_docs_count} new)")

        except Exception as e:
            old_words = df_original[df_original['Topic'] == topic_id]['Top_n_words'].iloc[0]
            new_top_words[topic_id] = [w.strip() for w in old_words.split(' - ')][:top_n]

    print(f"\n{'='*70}")
    print(f"Recalculation Summary")
    print(f"{'='*70}")
    print(f"Topics that received outliers    : {len(topics_to_recalc)}")
    print(f"Actually recalculated (high impact): {actually_recalculated}")
    print(f"Kept original (low impact)        : {len(topics_to_recalc) - actually_recalculated}")
    print(f"Total topics with original words  : {len(new_top_words) - actually_recalculated}")
    print(f"Preservation rate                 : {(len(new_top_words) - actually_recalculated) / len(new_top_words) * 100:.1f}%")
    print(f"{'='*70}")

    return new_top_words

# Recalculate with minimal approach
new_representative_words = recalculate_representative_words_minimal(
    df_updated,
    df,
    topics_with_new_docs,
    min_new_docs_ratio=0.15,
    top_n=10
)


Selective Recalculation
Minimum new docs ratio for recalc: 15%
  Topic 0: Impact 38.5%, RECALCULATED (5024 docs, +1935 new)
  Topic 1: Impact 31.1%, RECALCULATED (489 docs, +152 new)
  Topic 2: Impact 49.2%, RECALCULATED (595 docs, +293 new)
  Topic 3: Impact 51.5%, RECALCULATED (196 docs, +101 new)
  Topic 4: Impact 7.8% < threshold, KEEPING original
  Topic 5: Impact 59.1%, RECALCULATED (171 docs, +101 new)
  Topic 6: Impact 61.7%, RECALCULATED (162 docs, +100 new)
  Topic 7: Impact 43.8%, RECALCULATED (89 docs, +39 new)
  Topic 8: Impact 53.5%, RECALCULATED (71 docs, +38 new)
  Topic 9: Impact 34.7%, RECALCULATED (49 docs, +17 new)
  Topic 10: Impact 80.9%, RECALCULATED (162 docs, +131 new)
  Topic 11: Impact 81.4%, RECALCULATED (161 docs, +131 new)
  Topic 12: Impact 81.2%, RECALCULATED (160 docs, +130 new)
  Topic 13: Impact 55.2%, RECALCULATED (67 docs, +37 new)

Recalculation Summary
Topics that received outliers    : 14
Actually recalculated (high impact): 13
Kept original (lo

In [21]:
# Update DataFrame
for topic_id, words in new_representative_words.items():
    new_words_str = " - ".join(words)
    df_updated.loc[df_updated['Topic'] == topic_id, 'Top_n_words'] = new_words_str

metrics_after_recalc = evaluate_metrics(df_updated, "Final Results")


Evaluation: Final Results
Coherence C_v : 0.4744
Diversity     : 0.6214
Topics        : 14
Outliers      : 14 (0.19%)


In [22]:
df_updated.to_csv("after_reassign_indobert.csv", sep=";", index=False)

In [32]:
df_reassigned.to_csv('outlier_reassignment_details-bert.csv', index=False, encoding='utf-8-sig')
print("[3/7] outlier_reassignment_details-bert.csv")


[3/7] outlier_reassignment_details-bert.csv


In [23]:
summary = (
    df_updated.groupby('Topic')
      .agg(
          Total_Data=('Topic', 'count'),
          Top_N_Words=('Top_n_words', 'first')
      )
      .reset_index()
      .sort_values('Topic')
)

summary


,Topic,Total_Data,Top_N_Words
0,-1,14,outlier
1,0,5024,diri - sendiri - hidup - kecil - takut - salah...
2,1,489,terapi - psikolog - takut - khawatir - tahu - ...
3,2,595,gagal - takut - kecil - diri - sendiri - sukse...
4,3,196,otak - lupa - ingat - ulang - kecil - susah - ...
5,4,90,overthinking - henti - capek - overthinking ca...
6,5,171,ubah - kecil - hidup - stuck - diri - salah - ...
7,6,162,anak - salah - takut - kecil - gadget - dukung...
8,7,89,takut - dokter - mental - gila - obat - sehat ...
9,8,71,ulang - takut - cuci - kompulsif - ganggu - ce...


In [33]:
summary.to_csv("interpretation-table.csv", sep=";", index=False)

In [24]:
df_updated

,question,answer,from,filename,document_preprocessed,Topic,Probability,Top_n_words,Name,Document
0,Apa itu memvalidasi pikiran sendiri?,Cek data yang mendukung dan yang menolak keyak...,GPT,A1_GPT_EmotionalFirstAid23-45_50.json,memvalidasi sendiri data dukung tolak yakin ga...,10,0.251203,kuesioner - kecil - metode - kerja - diri - si...,-1_kecil_diri_sendiri_hidup,memvalidasi sendiri data dukung tolak yakin ga...
1,Kenapa aku malas mencoba lagi setelah jatuh?,"Kegagalan menciptakan ilusi ""hasilnya pasti sa...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,malas jatuh gagal cipta ilusi hasil alternatif...,0,0.673766,diri - sendiri - hidup - kecil - takut - salah...,0_diri_sendiri_kecil_hidup,malas jatuh gagal cipta ilusi hasil alternatif...
2,Bagaimana cara efektif meminta bantuan?,"Ungkap kebutuhan spesifik: ""Aku butuh teman br...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,efektif minta ungkap butuh spesifik butuh tema...,0,0.694161,diri - sendiri - hidup - kecil - takut - salah...,0_diri_sendiri_kecil_hidup,efektif minta ungkap butuh spesifik butuh tema...
3,Apa saja contoh faktor yang bisa kukendalikan?,"Jumlah waktu belajar harian, mentor yang dihub...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,contoh faktor kendali jumlah mentor target min...,10,1.000000,kuesioner - kecil - metode - kerja - diri - si...,10_kuesioner_metode_eksperimental_metode ekspe...,contoh faktor kendali jumlah mentor target min...
4,Bagaimana memecah tujuan besar agar realistis?,"Gunakan metode SMART-ER (Specific, Measurable,...",GPT,A1_GPT_EmotionalFirstAid23-45_50.json,pecah tuju realistis metode smart specific mea...,10,1.000000,kuesioner - kecil - metode - kerja - diri - si...,10_kuesioner_metode_eksperimental_metode ekspe...,pecah tuju realistis metode smart specific mea...
...,...,...,...,...,...,...,...,...,...,...
7495,Aku sering merasa orang lain nggak ngerti beta...,Pasti frustrasi kalau usahamu nggak diakui. Ta...,GROK,D9_GROK_EgoIsTheEnemy139-149_20.json,betapa usaha salah frustrasi usaha terlalu ene...,0,0.678011,diri - sendiri - hidup - kecil - takut - salah...,0_diri_sendiri_kecil_hidup,betapa usaha salah frustrasi usaha terlalu ene...
7496,Aku sering kesel sama tim karena mereka nggak ...,Setiap orang punya ritme kerja sendiri. Coba k...,GROK,D9_GROK_EgoIsTheEnemy184-194_20.json,kesel secepet salah ritme kerja sendiri target...,2,0.238851,gagal - takut - kecil - diri - sendiri - sukse...,2_gagal_takut gagal_takut_kecil,kesel secepet salah ritme kerja sendiri target...
7497,Kenapa aku susah buat konsisten sama tujuanku?,"Kadang, kita kehilangan fokus karena terlalu m...",GROK,D9_GROK_EgoIsTheEnemy39-49_20.json,susah konsisten tuju hilang terlalu mikirin ha...,2,0.273361,gagal - takut - kecil - diri - sendiri - sukse...,2_gagal_takut gagal_takut_kecil,susah konsisten tuju hilang terlalu mikirin ha...
7498,"Aku pengen jadi orang hebat, tapi kok kayaknya...",Jadi orang hebat nggak selalu tentang push dir...,GROK,D9_GROK_EgoIsTheEnemy61-71_20.json,jauh push diri mati mati push diri burnout kec...,5,0.229201,ubah - kecil - hidup - stuck - diri - salah - ...,5_ubah_kecil_ubah kecil_kecil ubah,jauh push diri mati mati push diri burnout kec...


## Extract Topic

In [27]:
import pandas as pd
import json
import os

# --- CONFIG ---
input_file = "after_reassign_indobert.csv"  # ganti dengan file CSV kamu
sep = ";"                # delimiter CSV
output_dir = "topik-extract"

os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_file, sep=sep)

required_cols = ['question', 'answer', 'Topic', 'Probability', 'Top_n_words']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan di CSV")

# --- LOOP PER TOPIC ---
for topic_id in df['Topic'].unique():
    df_topic = df[df['Topic'] == topic_id].copy()
    
    # Simpan CSV
    csv_file = os.path.join(output_dir, f"Topic_{topic_id}.csv")
    df_topic.to_csv(csv_file, sep=sep, index=False)

print("Selesai memisahkan data per topik!")


Selesai memisahkan data per topik!


In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 1. Load Data
file_path = r'C:\Users\Liza\Documents\Kerja Praktik\Topic modeling\berttopic\exp 7-bert\after_reassign_indobert.csv'
df = pd.read_csv(file_path, sep=';')

# 2. Mapping Topic IndoBERT (0-13)
topic_mapping = {
    0: "Kecemasan Diri dalam Kehidupan Sosial", 
    1: "Kecemasan terhadap Terapi dan Diagnosis Psikologis", 
    2: "Takut Gagal dan Keraguan Diri", 
    3: "Kesulitan Memori dan Strategi Belajar", 
    4: "Overthinking", 
    5: "Hidup Stuck dan Keinginan Berubah", 
    6: "Pola Pengasuhan Anak dan Kecemasan Orang Tua", 
    7: "Stigma dan Ketakutan Berobat", 
    8: "Pikiran Obsesif dan Perilaku Kompulsif", 
    9: "Mimpi, Imajinasi, dan Kreativitas", 
    10: "Pengelolaan diri dan Stres", 
    11: "Takut Gagal dalam Hidup & Kerja", 
    12: "Hubungan Otak dengan Gangguan Mental", 
    13: "Persepsi dan Ilusi"
}

def generate_scientific_wordcloud(df, topic_id):
    if topic_id not in df['Topic'].values:
        return

    # --- LANGKAH 1: Hitung c-TF-IDF Secara Presisi ---
    docs_per_topic = df.groupby('Topic')['document_preprocessed'].apply(lambda x: ' '.join(x)).reset_index()
    cv = CountVectorizer().fit(docs_per_topic.document_preprocessed)
    words = cv.get_feature_names_out()
    X = cv.transform(docs_per_topic.document_preprocessed).toarray()
    
    m = len(df)
    t = X.sum(axis=0)
    idf = np.log(1 + (m / t))
    
    topic_idx = docs_per_topic[docs_per_topic['Topic'] == topic_id].index[0]
    tf_topic = X[topic_idx]
    ctfidf_scores = tf_topic * idf

    # --- LANGKAH 2: Ambil Top N Word dari Kolom Excel/CSV ---
    known_top_words = df[df['Topic'] == topic_id]['Top_n_words'].iloc[0].split(' - ')
    
    word_score_map = {}
    for word_target in known_top_words:
        if word_target in words:
            idx = np.where(words == word_target)[0][0]
            word_score_map[word_target] = ctfidf_scores[idx]
        else:
            word_score_map[word_target] = 1

    # --- LANGKAH 3: Visualisasi ---
    wc = WordCloud(
        width=1000, height=500, 
        background_color='white',
        colormap='summer', # Warna cerah untuk tema kesehatan mental
        prefer_horizontal=0.85
    ).generate_from_frequencies(word_score_map)
    
    plt.figure(figsize=(12, 6))
    # Gunakan .to_image() untuk menghindari TypeError: asarray()
    plt.imshow(wc.to_image(), interpolation='bilinear')
    
    # Judul otomatis dari mapping
    sub_tema = topic_mapping.get(topic_id, "Tema Umum")
    plt.title(f"Topik {topic_id}: {sub_tema}", fontsize=18, pad=20, fontweight='bold')
    plt.axis("off")
    
    # Simpan Gambar dengan kualitas tinggi
    output_filename = f'wordcloud_topic_{topic_id}.png'
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"File disimpan: {output_filename}")

# --- EKSEKUSI LOOPING OTOMATIS 0-25 ---
print("Sedang memproses 14 Word Cloud berdasarkan skor c-TF-IDF...")
for i in range(14):
    generate_scientific_wordcloud(df, i)
print("\nSemua Word Cloud berhasil dibuat dan disimpan!")

Sedang memproses 14 Word Cloud berdasarkan skor c-TF-IDF...
File disimpan: wordcloud_topic_0.png
File disimpan: wordcloud_topic_1.png
File disimpan: wordcloud_topic_2.png
File disimpan: wordcloud_topic_3.png
File disimpan: wordcloud_topic_4.png
File disimpan: wordcloud_topic_5.png
File disimpan: wordcloud_topic_6.png
File disimpan: wordcloud_topic_7.png
File disimpan: wordcloud_topic_8.png
File disimpan: wordcloud_topic_9.png
File disimpan: wordcloud_topic_10.png
File disimpan: wordcloud_topic_11.png
File disimpan: wordcloud_topic_12.png
File disimpan: wordcloud_topic_13.png

Semua Word Cloud berhasil dibuat dan disimpan!
